#Imports

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import isnull, count, when
import requests
from pyspark.sql import SparkSession
from sedona.register.geo_registrator import SedonaRegistrator
import geopandas as gpd
from sedona.utils.adapter import Adapter


from io import StringIO

In [3]:
import geopandas as gpd
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType
from shapely.geometry import Point

# Caminho para o shapefile
shapefile_path = "../NYC_Borough_Boundary_6403672305752144374/NYC_Borough_Boundary.shp"

# Carregar shapefile com reprojeção para EPSG:4326
gdf = gpd.read_file(shapefile_path).to_crs(epsg=4326)

# Criar dicionário de boroughs com geometria correta
borough_shapes = {
    row['BoroName']: row['geometry']
    for _, row in gdf.iterrows()
}

# Bounding boxes (caso queira)
borough_bboxes = {
    name: geom.bounds
    for name, geom in borough_shapes.items()
}

# Função corrigida
def get_borough_name(lat, lon):
    point = Point(lon, lat)
    for name, polygon in borough_shapes.items():
        if polygon.contains(point):
            return name
    return None

get_borough_name_udf = udf(get_borough_name, StringType())

C:\Users\Alan-\Desktop\AC2_BigData\venv\Lib\site-packages\pyogrio\raw.py:198: RuntimeWarning: ../NYC_Borough_Boundary_6403672305752144374/NYC_Borough_Boundary.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return ogr_read(


#Data Initialization

In [4]:
import sys
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .config("spark.pyspark.python", sys.executable) \
    .config("spark.pyspark.driver.python", sys.executable) \
    .appName("sparksubmit_test_app") \
    .getOrCreate()

In [5]:
df = spark.read.csv("../yellow_tripdata_2015-01.csv", header=True, inferSchema=True)

#Exploratory Data Analysis + Data-Preprocessing

In [ ]:
df.show()

+--------+--------------------+---------------------+---------------+-------------+------------------+------------------+----------+------------------+------------------+------------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|  pickup_longitude|   pickup_latitude|RateCodeID|store_and_fwd_flag| dropoff_longitude|  dropoff_latitude|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|
+--------+--------------------+---------------------+---------------+-------------+------------------+------------------+----------+------------------+------------------+------------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+
|       2| 2015-01-15 19:05:39|  2015-01-15 19:23:42|              1|         1.59|  -73.993896484375|  40.7501106262207|         1|    

In [ ]:
num_linhas = df.count()
print(f"Número de linhas no DataFrame: {num_linhas}")

Número de linhas no DataFrame: 12748986


In [ ]:
# Exibir o esquema do DataFrame
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- RateCodeID: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)



In [ ]:
df.summary().show()

+-------+------------------+------------------+------------------+-------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+-------------------+------------------+------------------+-------------------+---------------------+------------------+
|summary|          VendorID|   passenger_count|     trip_distance|   pickup_longitude|   pickup_latitude|        RateCodeID|store_and_fwd_flag| dropoff_longitude|  dropoff_latitude|      payment_type|       fare_amount|              extra|           mta_tax|        tip_amount|       tolls_amount|improvement_surcharge|      total_amount|
+-------+------------------+------------------+------------------+-------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+-------------------+------------------+------------------+-------------------+---------------------+---

In [ ]:
df.select([count(when(isnull(c), c)).alias(c) for c in df.columns]).show()

+--------+--------------------+---------------------+---------------+-------------+----------------+---------------+----------+------------------+-----------------+----------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|pickup_longitude|pickup_latitude|RateCodeID|store_and_fwd_flag|dropoff_longitude|dropoff_latitude|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|
+--------+--------------------+---------------------+---------------+-------------+----------------+---------------+----------+------------------+-----------------+----------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+
|       0|                   0|                    0|              0|            0|               0|              1|         1|                 1|              

In [6]:
df_payment_type_1 = df.filter(df["payment_type"] == 1)
df_payment_type_1.show()

+--------+--------------------+---------------------+---------------+-------------+------------------+------------------+----------+------------------+------------------+------------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|  pickup_longitude|   pickup_latitude|RateCodeID|store_and_fwd_flag| dropoff_longitude|  dropoff_latitude|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|
+--------+--------------------+---------------------+---------------+-------------+------------------+------------------+----------+------------------+------------------+------------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+
|       2| 2015-01-15 19:05:39|  2015-01-15 19:23:42|              1|         1.59|  -73.993896484375|  40.7501106262207|         1|    

In [7]:
df_with_borough = df_payment_type_1.withColumn(
    "pickup_borough", get_borough_name_udf("pickup_latitude", "pickup_longitude")
).drop("pickup_longitude", "pickup_latitude")

df_with_borough = df_with_borough.withColumn(
    "dropoff_borough", get_borough_name_udf("dropoff_latitude", "dropoff_longitude")
).drop("dropoff_longitude", "dropoff_latitude")

# 6. Mostrar resultado
df_with_borough.show()

PythonException: 
  An exception was thrown from the Python worker. Please see the stack trace below.
Traceback (most recent call last):
  File "C:\Users\Alan-\Desktop\AC2_BigData\venv\Lib\site-packages\pyspark\python\lib\pyspark.zip\pyspark\worker.py", line 1100, in main
pyspark.errors.exceptions.base.PySparkRuntimeError: [PYTHON_VERSION_MISMATCH] Python in worker has different version (3, 10) than that in driver 3.13, PySpark cannot run with different minor versions.
Please check environment variables PYSPARK_PYTHON and PYSPARK_DRIVER_PYTHON are correctly set.
